In [ ]:
# Now we practise writing code without AI Agents help; we have gotten way stronger than before; you just need 
# to confidently try implementing these functions; its all just simple trial and error and implementations
# Play Bold My G

# Below is the implementation for RAG Code line by line



import numpy as np
import faiss

from sentence_transformers import SentenceTransformers


documents = [
    "Transformers use self-attention mechanisms.",
    "RAG combines retrieval with generation.",
    "FAISS enables efficient vector similarity search.",
    "Embeddings convert text into dense vectors.",
    "Chunking improves retrieval quality."
]


def chunk_text(text, chunk_size=200):
    words = text.split()

    chunks = []

    for i in range(0,len(words),chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


    
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc)
    all_chunks.extand(chunks)



model = SentenceTransformers(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = model.encode(all_chunks)
embeddings = np.array(embeddings).astype("float32")


dimension = embeddings.shape[1]


def retrieve(query,top_k =2):
    query_embedding =model.encode([query])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    result = []

    for idx in indices[0]:
        results.append(all_chunks[idx])


    return results



# the generative part, ie all of aboves invocation

query = "WHat is RAG"

retrieved_chunks = retrieve(query)

context = "\n".join(retrieved_chunks)


prompt = f"""
Answer the question based on the context below

context:
{context}

question:
{query}

"""

print(prompt)



ModuleNotFoundError: No module named 'faiss'

In [6]:
# overlapping chunking

def chunkoverlap(text,chunk_size=200,overlap=50):
    chunks = []

    words = text.split()

    step = chunk_size - overlap
    for i in range(0,len(words), step):
        chunk = " ".join(
            words[i:i+chunk_size]
        )

        chunks.append(chunk)

    return chunks


# Retrieveing even similarity scores

def retrieve(query, top_k = 2):
    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances , indices = index.search(query_embedding,top_k)

    result = []

    for score, idx in zip(distances[0], indices[0]):

        result.append({
            "chunk": all_chunks[idx],
            "score": float(score)

        })

    return result

# Dirty kiddish RAG is when you do all chunks in a list; we want to give our chunks more context and hence we make them disctionaries adding more metadata to them

# what i mean, example

all_chunks = [
    {
        "text": "Transformers use attention",
        "Source": "Paper1.pdf",
        "Page": 3
    },
    {
        "text":"RAG Combines retrieval and generation",
        "Source": "doc2.pdf",
        "page":5
    }
]



all_chunks = [
    "some chunks"
]





In [7]:
# NOW the full approach of chunking of documents with metadata

documents = [
    {
        "text": "Transformers use self-attention.",
        "source": "doc1.pdf",
        "page": 1
    },
    {
        "text": "RAG combines retrieval and generation.",
        "source": "doc2.pdf",
        "page": 5
    }
]


all_chunks = []

for doc in documents:

    chunks = chunk_text(doc["text"])

    for chunk in chunks:

        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"]
        })



NameError: name 'chunk_text' is not defined

In [ ]:
def chunk_text_overlap(
    text,
    chunk_size=200,
    overlap=50
):

    words = text.split()

    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        chunks.append(chunk)

    return chunks

def chunks_with_metadata(
        documents,
        chunk_size=200,
        overlap = 50
        ):
    
    all_chunks = []

    for doc in documents:

        chunks = chunk_text_overlap(
            text=doc["text"],
            chunk_size=chunk_size,
            overlap=overlap
        )

        for chunk in chunks:
            all_chunks.append({
                'text':chunk,
                'source':doc['source'],
                'page':doc['page']
            })

    return all_chunks
    

# once all your chunks are now in dictionary forms with metadata; you need to vary even the embedding step to handle the chunks data coming from it

texts = [chunk["text"] for chunk in all_chunks]

embeddings = model.encode(texts)









In [ ]:
# Some spark syntax practise

df = spark.read.paraquet(
    "s3://bucket/carriers/"
)

df = spark.read.csv(
     "s3://bucket/carriers/",
     header = True,
     inferschema = True

)

df = df.filter(
    col("violations")>10
)

# Mutiple conditions
df = df.filter(
    (col("VIolations")>10)
    &
    (col("Crashes")>0)
)


df = df.select(
    "carrier_id",
    "violations",
    "crashes",
)


df = df.withcolumns(
    "risk_score",
    col
)

from pyspark.sql.functions import sum

df.groupBy(
    "carrier_id"
).agg(
    sum("violations").alias(
        "total_violations"
    )
)

df.groupBy(
    "carrier_id"
).agg(
    sum("violations").alias(
        "total_violations"
    )
)




from pyspark.sql.functions import (sum,avg,max)

df.groupBy(
    "carrier_id"
).agg(
    sum("violations"),
    max("crashes"),
    max("crashes")
)


df = df.repartition(
    200,
    "carrier_id"
)

#repartitioning is useful to do before groupby's and all


df = df.coalesce(10)
#this is very common before writing the output


df.cache()   #this is to store data in memory so we dont have to recompute
df.count()   #this is materializing it


from pyspark import storagelevel

df.persist(
    storagelevel.MEMORY_AND_DISK
)


df = df.dropDuplicates(
    ['carrier_id']
)

df = df.orderby(
    "violations"
)


from pyspark.sql.functions import desc

df.orderBy(
    desc("violations")
)



from pyspark.sql.window import Window

from pyspark.sql.functions import (
    rank
)

window = Window.partitionBy(
    "state"
).orderBy(
    col("violations").desc()
)

df = df.withColumn(
    "rank",
    rank().over(window)
)



from pyspark.sql.window import Window

from pyspark.sql.functions import avg

window = Window.partitionBy(
    "carrier_id"
).orderBy(
    "date"
)

df = df.withColumn(
    "rolling_avg",
    avg("violations").over(window)
)



df.explain()

# Now lets see some user defined functions (UDF) in spark

from pyspark.sql.functions import udf

from pyspark.sql.types import IntegerType

def rick_calc(v):

    return v*2

risk_udf = udf(
    risk_calc,
    IntegerType()
)

df=df.withcolumn(
    "risk",
    risk_udf(
        col('violations')
    )
)

from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def multiply_by_two(s):

    return s * 2




spark.conf.set(
    "spark.sql.adaptive.enabled",
    "true"
)

In [ ]:
# now for some streaming syntax

from pyspark.sql.window import Window

stream_df = spark.readstream.format(
    "kafka"
).load()

stream_df = stream_df.withwatermark(
    "event_time",
    "10 minutes"
)



In [ ]:
# Deepgram connection example

import asyncio
import websockets

async def hello():
    async with websockets.connect(
        "ws://localhost:8000"
    ) as websocket:
        await websocket.send(
            "hello"
        )

        response = await websocket.recv()

        print(response)


asyncio.run(
    hello()
)

In [ ]:
# basic stripe code to showcase webhooks


from fastapi import fastapi
from fastapi import request

app = FastAPI()

@app.post(
    "stripe/webhook"
)

async def stripe_webhook(
    request:request
):
    payload = await request.json()

    print(payload)

    return {
        "status": "ok"
    }

In [ ]:
from fastapi import FastAPI, Header, HTTPException

app = FastAPI()

API_KEY = "abc123"


@app.post("/predict")
async def predict(
    x_api_key: str = Header()
):

    if x_api_key != API_KEY:

        raise HTTPException(
            status_code=401
        )

    return {
        "prediction": 0.87
    }

In [ ]:
import stripe

from fastapi import (
    FastAPI,
    Request,
    HTTPException
)

app = FastAPI()

WEBHOOK_SECRET = (
    "whsec_abc123"
)


@app.post("/stripe/webhook")
async def stripe_webhook(
    request: Request
):

    payload = await request.body()

    signature = request.headers.get(
        "stripe-signature"
    )

    try:

        event = (
            stripe.Webhook.construct_event(
                payload,
                signature,
                WEBHOOK_SECRET
            )
        )

    except Exception:

        raise HTTPException(
            status_code=400
        )

    if (
        event["type"]
        ==
        "checkout.session.completed"
    ):

        print("Payment successful")

    return {"received": True}